# Cucker–Smale Flocking: Emergent Consensus in Multi-Agent Systems

Flocks of birds, schools of fish, and swarms of insects exhibit striking collective behaviors without any central coordinator. Each individual follows only simple local rules involving its neighbors, yet the group as a whole moves in a coherent, organized fashion. This **emergence of order from local interactions** is the central puzzle of collective motion.

## The Cucker–Smale model

Cucker and Smale (2007) proposed a mathematically tractable model of flocking. $n$ particles have positions $x_i(t) \in \mathbb{R}^d$ and velocities $v_i(t) \in \mathbb{R}^d$. The dynamics are:
$$
\dot{x}_i = v_i, \qquad \dot{v}_i = \frac{K}{n} \sum_{j=1}^n \psi(|x_i - x_j|)(v_j - v_i),
$$
where $\psi : \mathbb{R}_{\geq 0} \to \mathbb{R}_{\geq 0}$ is a **communication weight** that decreases with distance, and $K > 0$ is a coupling strength. The velocity update pushes particle $i$'s velocity toward the average velocity of its neighbors, weighted by influence $\psi$.

## Communication function

The canonical choice is
$$
\psi(r) = \frac{1}{\bigl(1 + (r/r_0)^2\bigr)^\beta},
$$
with range parameter $r_0 > 0$ and decay exponent $\beta \geq 0$. For $\beta = 0$, all pairs interact equally (mean-field). As $\beta \to \infty$, only nearest neighbors interact (topological flocking).

## Flocking theorem

Define the **velocity variance** $e(t) = \tfrac{1}{2n}\sum_i |v_i - \bar{v}|^2$ and **diameter** $d(t) = \max_{i,j}|x_i - x_j|$. Cucker and Smale proved:
- If $\beta < 1/2$ (**strong communication**), the system achieves **unconditional flocking**: $e(t) \to 0$ for all initial conditions.
- If $\beta \geq 1/2$ (**weak communication**), flocking may still occur depending on the initial configuration.

## Laplacian formulation

In matrix form, the velocity ODE reads $\dot{V} = K \cdot L(X) \cdot V$, where $V$ is the $n \times d$ velocity matrix and $L(X)$ is the **influence Laplacian**:
$$
L_{ij}(X) = \begin{cases} \psi(|x_i - x_j|)/n & i \neq j, \\ -\sum_{k \neq i} \psi(|x_i - x_k|)/n & i = j. \end{cases}
$$
This is a negative semi-definite matrix with a zero eigenvalue corresponding to the consensus direction, connecting flocking to **consensus algorithms** in control theory.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams["figure.dpi"] = 120

## Simulation engine

We use a discrete-time Euler integration of the Cucker–Smale ODE. The influence Laplacian $L(X)$ is recomputed at every time step since positions evolve. We work in $\mathbb{C} \cong \mathbb{R}^2$ for notational convenience (positions and velocities are complex numbers).

In [ ]:
def psi_func(r, r0=0.7, beta=0.6):
    """Communication weight psi(r) = 1/(1+(r/r0)^2)^beta."""
    return 1.0 / (1.0 + (r / r0) ** 2) ** beta


def influence_laplacian(x, r0=0.7, beta=0.6):
    """Build n x n influence Laplacian for positions x (complex array)."""
    n = len(x)
    D = np.abs(x[:, None] - x[None, :])  # distance matrix
    W = psi_func(D, r0, beta) / n        # weight matrix
    np.fill_diagonal(W, 0.0)
    L = W - np.diag(W.sum(axis=1))
    return L


def run_flocking(n=80, n_steps=100, tau=1.3, K=0.2,
                 r0=0.7, beta=0.6, seed=42):
    """
    Simulate Cucker-Smale flocking.
    Returns:
        xs: (n_steps, n) complex position array
        vs: (n_steps, n) complex velocity array
    """
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(n) + 1j * rng.standard_normal(n)
    v = (rng.standard_normal(n) + 1j * rng.standard_normal(n)) * 0.06 + 0.02

    xs = np.zeros((n_steps, n), dtype=complex)
    vs = np.zeros((n_steps, n), dtype=complex)

    for t in range(n_steps):
        xs[t] = x
        vs[t] = v
        L = influence_laplacian(x, r0, beta)
        x_new = x + tau * v
        v_new = v + tau * K * (L @ v)
        x, v = x_new, v_new

    return xs, vs


print("Flocking simulation ready.")

## Velocity consensus over time

We first quantify the degree of flocking via the **velocity variance**:
$$
e(t) = \frac{1}{2n} \sum_{i=1}^n |v_i(t) - \bar{v}(t)|^2, \qquad \bar{v}(t) = \frac{1}{n}\sum_i v_i(t).
$$
For a fully synchronized flock, $e(t) \to 0$. We also track the **spatial diameter** $d(t) = \max_{i,j} |x_i - x_j|$.

In [ ]:
n_steps = 120
xs, vs = run_flocking(n=80, n_steps=n_steps, tau=1.3, K=0.2,
                      r0=0.7, beta=0.6, seed=42)

v_mean = vs.mean(axis=1, keepdims=True)  # (n_steps, 1)
e_t = 0.5 * np.mean(np.abs(vs - v_mean)**2, axis=1)
d_t = np.array([np.max(np.abs(xs[t][:, None] - xs[t][None, :]))
                for t in range(n_steps)])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].semilogy(e_t, 'b-', lw=2)
axes[0].set_xlabel("time step"); axes[0].set_ylabel("velocity variance $e(t)$")
axes[0].set_title("Velocity consensus: $e(t) \\to 0$"); axes[0].grid(alpha=0.3)

axes[1].plot(d_t, 'tomato', lw=2)
axes[1].set_xlabel("time step"); axes[1].set_ylabel("spatial diameter $d(t)$")
axes[1].set_title("Spatial spreading of the flock"); axes[1].grid(alpha=0.3)

plt.suptitle("Cucker–Smale flocking: velocity variance and diameter", y=1.02)
plt.tight_layout()
plt.show()

## Visualization of particle trajectories

We draw the positions of all $n$ particles at four time steps: early (scattered and incoherent), transitional, and late (aligned flock). The velocity arrows show each particle's direction of motion. Colors transition from blue (early) to red (late).

In [ ]:
frames = [0, 10, 30, 80]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, t in zip(axes, frames):
    x_t = xs[t]
    v_t = vs[t]
    color = t / n_steps
    col = (color, 0.0, 1 - color)

    ax.scatter(x_t.real, x_t.imag, s=20, color=col, zorder=5)
    # velocity arrows (scaled)
    v_scale = 2.0
    for xi, vi in zip(x_t, v_t):
        ax.annotate("",
                    xy=(xi.real + v_scale*vi.real, xi.imag + v_scale*vi.imag),
                    xytext=(xi.real, xi.imag),
                    arrowprops=dict(arrowstyle='->', color=col, lw=0.8))

    ax.set_title(f"$t = {t}$,  $e={e_t[t]:.4f}$", fontsize=9)
    ax.set_aspect('equal')
    # Determine axis limits from data
    xr = x_t.real; xi_arr = x_t.imag
    margin = max(1.0, (xr.max()-xr.min())*0.2, (xi_arr.max()-xi_arr.min())*0.2)
    ax.set_xlim(xr.mean()-4, xr.mean()+4)
    ax.set_ylim(xi_arr.mean()-4, xi_arr.mean()+4)
    ax.grid(alpha=0.2); ax.axis('off')

fig.suptitle("Particle positions and velocities: emergence of collective motion", y=1.02)
plt.tight_layout()
plt.show()

## Effect of communication range $r_0$ and decay $\beta$

The **range** $r_0$ controls how far the influence extends: large $r_0$ means all particles interact (global coupling), small $r_0$ means only close neighbors interact (local coupling). The **decay exponent** $\beta$ controls how fast the influence drops with distance. The flocking theorem guarantees convergence for $\beta < 1/2$ regardless of initial conditions.

In [ ]:
configs_psi = [
    (0.3, 0.6, "small range $r_0=0.3$"),
    (0.7, 0.6, "medium range $r_0=0.7$"),
    (2.0, 0.6, "large range $r_0=2.0$"),
    (0.7, 1.5, r"strong decay $\beta=1.5$"),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors_c = plt.cm.tab10(np.linspace(0, 0.4, len(configs_psi)))

r_plot = np.linspace(0, 4, 300)
for (r0, beta, lbl), col in zip(configs_psi, colors_c):
    axes[0].plot(r_plot, psi_func(r_plot, r0, beta), lw=2, color=col, label=lbl)

axes[0].set_xlabel("distance $r$"); axes[0].set_ylabel(r"$\psi(r)$")
axes[0].set_title("Communication weight $\\psi(r)$")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

for (r0, beta, lbl), col in zip(configs_psi, colors_c):
    _, vs_i = run_flocking(n=60, n_steps=100, r0=r0, beta=beta, seed=42)
    vm_i = vs_i.mean(axis=1, keepdims=True)
    e_i = 0.5 * np.mean(np.abs(vs_i - vm_i)**2, axis=1)
    axes[1].semilogy(e_i, lw=2, color=col, label=lbl)

axes[1].set_xlabel("time step"); axes[1].set_ylabel("velocity variance $e(t)$")
axes[1].set_title("Convergence to consensus"); axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle("Effect of communication parameters on flocking speed", y=1.02)
plt.tight_layout()
plt.show()

## Interactive flocking simulation

Vary the coupling strength $K$ and the communication range $r_0$ to see how they affect the dynamics. The plot shows the final configuration (positions and velocity arrows) alongside the velocity variance over time.

In [ ]:
def show_flocking(K=0.2, r0=0.7, beta=0.6, n_steps=100):
    xs_i, vs_i = run_flocking(n=60, n_steps=n_steps, tau=1.3,
                               K=K, r0=r0, beta=beta, seed=42)
    vm_i = vs_i.mean(axis=1, keepdims=True)
    e_i  = 0.5 * np.mean(np.abs(vs_i - vm_i)**2, axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    x_final = xs_i[-1]; v_final = vs_i[-1]
    axes[0].scatter(x_final.real, x_final.imag, s=25, c='royalblue', zorder=5)
    v_med = np.median(np.abs(v_final))
    scale = 1.5 / (v_med + 1e-8)
    for xi, vi in zip(x_final, v_final):
        axes[0].annotate("",
                         xy=(xi.real + scale*vi.real, xi.imag + scale*vi.imag),
                         xytext=(xi.real, xi.imag),
                         arrowprops=dict(arrowstyle='->', color='navy', lw=0.8))
    axes[0].set_aspect('equal'); axes[0].grid(alpha=0.2)
    axes[0].set_title(f"Final state ($t={n_steps}$, $e={e_i[-1]:.4f}$)")
    xr = x_final.real; xi_arr = x_final.imag
    c_x, c_y = xr.mean(), xi_arr.mean()
    r_max = max(4, xr.std()*4, xi_arr.std()*4)
    axes[0].set_xlim(c_x-r_max, c_x+r_max)
    axes[0].set_ylim(c_y-r_max, c_y+r_max)

    axes[1].semilogy(e_i, 'b-', lw=2)
    axes[1].set_xlabel("time step"); axes[1].set_ylabel("$e(t)$")
    axes[1].set_title("Velocity variance")
    axes[1].grid(alpha=0.3)

    plt.suptitle(fr"Cucker–Smale:  $K={K}$,  $r_0={r0}$,  $\beta={beta}$", y=1.02)
    plt.tight_layout(); plt.show()

interact(
    show_flocking,
    K=FloatSlider(value=0.2, min=0.02, max=1.0, step=0.02, description="$K$"),
    r0=FloatSlider(value=0.7, min=0.1, max=3.0, step=0.1, description="$r_0$"),
    beta=FloatSlider(value=0.6, min=0.0, max=2.0, step=0.1, description=r"$\beta$"),
    n_steps=IntSlider(value=100, min=20, max=200, step=10, description="steps"),
);

## Bibliographical resources

- Cucker, F. and Smale, S. (2007). Emergent behavior in flocks. *IEEE Transactions on Automatic Control*, 52(5), 852–862.
- Cucker, F. and Smale, S. (2007). On the mathematics of emergence. *Japanese Journal of Mathematics*, 2(1), 197–227.
- Vicsek, T., Czirók, A., Ben-Jacob, E., Cohen, I. and Shochet, O. (1995). Novel type of phase transition in a system of self-driven particles. *Physical Review Letters*, 75(6), 1226.
- Ha, S.-Y. and Liu, J.-G. (2009). A simple proof of the Cucker–Smale flocking dynamics and mean-field limit. *Communications in Mathematical Sciences*, 7(2), 297–325.
- Motsch, S. and Tadmor, E. (2014). Heterophilious dynamics enhances consensus. *SIAM Review*, 56(4), 577–621.